# HABS Neuro-AI Hackathon 2026: Team 12 results walkthrough

This notebook reproduces the main tables and four publication figures from the verified aggregate outputs in `results/`. All numbers come from nested person-by-theme cross-validation, fold-isolated scaling, and forward-only post-release filtering across 314 participants and 22,974 trials.

**Research question:** How do cortical activity (EEG) and peripheral autonomic responses (ECG, EDA, respiration) unfold during acute distress, does early EEG prospectively forecast future autonomic reactivity beyond past body dynamics, and what limits passive multimodal stress monitoring across unseen people, unseen videos, days, and cohorts?


In [1]:
import json
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

from habs_stress.figures import generate_all

ROOT = Path('..').resolve() if Path('../results').is_dir() else Path('.').resolve()
RESULTS = ROOT / 'results'
FIGS = ROOT / 'paper' / 'figures'

ph = json.loads((RESULTS / 'physiology.json').read_text())
cf = json.loads((RESULTS / 'causal_forecasting.json').read_text())
gc = json.loads((RESULTS / 'generalization_clinical.json').read_text())
print('Loaded verified aggregates from:', RESULTS)


## 1. Acute stress physiology and ECG morphology

Ninety-six point six percent of Lead I ECG recordings are inverted. After automatic polarity correction and beat-by-beat P-QRS-T delineation, acute stress shows RR lengthening, a null PR segment, and a Bazett QTc drop that disappears once delta-RR is controlled, consistent with vagally mediated cardiac deceleration. Raw CURATED heartbeat-evoked potentials match ICA-preprocessed potentials and remain null.


In [2]:
paths = generate_all(RESULTS, FIGS)
display(Image(filename=str(FIGS / 'fig1_physiology_cascade.png'), width=900))


## 2. Prospective early EEG to future physiology (t = 5.000 s boundary)

No post-release sample at or after t = 5.000 s touches any predictor. Early EEG (REVE-base) improves future heart rate prediction over past body dynamics (M0 r = 0.6849, M2_joint +0.0110, 71.4% of people improved, replicated in 5/5 cohorts under LOCO-CV: +0.0107). It survives nine artefact covariates, 46 interior non-rim channels, clean trials, and same-video other-viewer controls (+0.0242). No incremental benefit was detected for skin conductance (-0.0097) or respiration (+0.0008).


In [3]:
pri = cf['audit_forecast']['module2_primary_forecasting']['primary_cold_start_3x3']
rows = []
for name, key in [
    ('Heart rate [5, 10) s', 'tgt_ecg_hr_chg_5_10'),
    ('Skin conductance [5, 10) s', 'tgt_eda_sc_chg_5_10'),
    ('Respiration [5, 10) s', 'tgt_resp_amp_chg_5_10'),
]:
    rows.append({
        'Target': name,
        'M0 body-only (r)': round(pri[key]['models']['M0_linear']['r']['mean'], 4),
        'M2 resid minus M0': round(pri[key]['paired_vs_m0_linear']['M2_reve_resid']['delta_r']['mean'], 4),
        'M2 joint minus M0': round(pri[key]['paired_vs_m0_linear']['M2_reve_joint']['delta_r']['mean'], 4),
        'GBDT M2 minus M0': round(pri[key]['paired_m2_gbdt_vs_m0_gbdt']['delta_r']['mean'], 4),
    })
display(pd.DataFrame(rows))
display(Image(filename=str(FIGS / 'fig2_causal_brain_heart.png'), width=900))


## 3. Generalization ladder, cross-day drift, and k-shot personalization

Moving from random splits (R1: 0.5887) to unseen people and unseen themes (R4: 0.5688) removes 0.0199 AUC, 70% of which comes from seen video themes. Across a one-week gap, EDA-only decoding drops by 0.039 AUC while EEG and multimodal models hold steady. Naive personal retraining collapses at k = 5 (-0.081 AUC), whereas Empirical-Bayes MAP shrinkage avoids collapse.


In [4]:
display(Image(filename=str(FIGS / 'fig3_generalization_drift.png'), width=950))


## 4. Subjective-autonomic divergence and simulated passive monitoring

Varimax decomposition across 314 participants (16,276 complete trials) separates skin conductance (F1), automatic cortico-cardiorespiratory activity (F2), and subjective-cardiorespiratory ratings (F3). Twenty-one percent of participants show null or negative subjective-physiological coherence. Aggregating discrete Cold-Start episode scores from K = 1 (10 s) to K = 10 (100 s cumulative; disjoint without-replacement blocks verified for K <= 5) raises AUC to 0.680-0.698 and doubles sensitivity at 90% specificity (21% to 42-53%).


In [5]:
m10e = gc['module10e_clinical_monitoring']
mon_rows = []
for lbl, key in [
    ('Standard stress vs. other', 'stress_standard_vs_nonstress'),
    ('Acute threat vs. calm/neutral', 'acute_threat_vs_calm_neutral'),
    ('Strict stress (3/7) vs. relaxed', 'stress_strict37_vs_relaxed64'),
]:
    bk = m10e[key]['S07_enriched']['by_k']
    mon_rows.append({
        'Clinical contrast': lbl,
        '10 s AUC': round(bk['1']['auc']['mean'], 4),
        '100 s AUC': round(bk['10']['auc']['mean'], 4),
        '10 s Sens@90%Spec': f"{100 * bk['1']['sens_at_spec90']['mean']:.1f}%",
        '100 s Sens@90%Spec': f"{100 * bk['10']['sens_at_spec90']['mean']:.1f}%",
    })
display(pd.DataFrame(mon_rows))
display(Image(filename=str(FIGS / 'fig4_coherence_monitoring.png'), width=900))
